# ⚡ Apache Spark: Transformations & Actions, Explained Simply

Welcome! This notebook teaches the **single most important idea in Spark**:

> **Transformations** *describe* what you want to do. **Actions** *actually do it.*

**What you'll learn**

1. 🧠 The big idea: lazy evaluation
2. 🔧 Setting up Spark
3. 📦 RDDs: transformations and actions
4. 📊 DataFrames: transformations and actions
5. 🔀 Narrow vs wide transformations (shuffles)
6. 💾 Caching
7. 🏁 A mini end-to-end example (word count)
8. 📝 Cheat sheet and practice exercises

Run the cells from top to bottom. Every code cell is small and has a comment on each line that matters.

## 1. 🧠 The Big Idea: Lazy Evaluation

Think of ordering at a restaurant:

| Restaurant | Spark |
|---|---|
| You tell the waiter your order (starter, main, dessert) | You write **transformations** (`filter`, `map`, `select`...) |
| Nothing is cooked yet, the waiter just writes it down | Nothing runs yet, Spark just builds a **plan** (called a *DAG*) |
| You say *"Please bring the food!"* | You call an **action** (`count`, `show`, `collect`...) |
| The kitchen cooks everything, in the smartest order | Spark runs the whole plan, optimised |

### Transformations vs Actions at a glance

| | **Transformation** | **Action** |
|---|---|---|
| What it does | Builds a new dataset from an old one | Triggers computation and returns a result |
| Runs immediately? | ❌ No (lazy) | ✅ Yes |
| Returns | A new RDD / DataFrame | A Python value (number, list...) or writes data |
| Examples | `map`, `filter`, `select`, `groupBy` | `count`, `collect`, `show`, `take` |

**Why lazy?** Because Spark can look at your *whole* plan first and skip unnecessary work. For example, if you only ask for the first 5 results, it does not need to process everything.

**Another key fact:** data in Spark is **immutable**. A transformation never changes the original, it always returns a *new* dataset.

## 2. 🔧 Setup

Install PySpark if you don't have it (uncomment the line below). You also need Java 8, 11, 17 or 21 installed.

The **`SparkSession`** is your entry point to everything in Spark. We run in `local[*]` mode, meaning Spark uses all CPU cores of your own computer, so no cluster is needed.

In [ ]:
# !pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("TransformationsAndActions")
    .master("local[*]")                        # run locally using all cores
    .config("spark.sql.shuffle.partitions", 4) # small number: good for tiny demo data
    .getOrCreate()
)

sc = spark.sparkContext          # the lower-level entry point, used for RDDs
sc.setLogLevel("ERROR")          # keep the output clean

print("Spark version:", spark.version)

## 3. 📦 Part A: RDDs (Resilient Distributed Datasets)

An **RDD** is Spark's original, low-level data structure: a collection of items split into **partitions** spread over many machines (or cores).

We start with RDDs because they show the transformation/action idea in its purest form.

### 3.1 Creating an RDD

`parallelize` turns a normal Python list into an RDD. `numSlices` is the number of partitions.

In [ ]:
numbers = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], numSlices=2)

print(numbers)                 # just a description, no data is shown!
print("Partitions:", numbers.getNumPartitions())

Notice that printing the RDD did **not** show the numbers. An RDD is only a *recipe*. To see data, we need an **action**.

---
### 3.2 🔧 RDD Transformations (lazy)

These all return a **new RDD** and run **nothing** yet.

#### `map(f)`: change every element
Applies a function to each element, one output per input.

In [ ]:
squared = numbers.map(lambda x: x * x)   # transformation: nothing runs yet
print(squared)                            # still just an RDD object

#### `filter(f)`: keep only some elements
Keeps elements for which the function returns `True`.

In [ ]:
even = numbers.filter(lambda x: x % 2 == 0)   # transformation

#### `flatMap(f)`: map, then flatten
Each input can produce **zero, one or many** outputs. Great for splitting sentences into words.

```
"hello world"  --map-->      ["hello", "world"]      (a list inside a list)
"hello world"  --flatMap-->   "hello", "world"       (flattened)
```

In [ ]:
sentences = sc.parallelize(["hello spark world", "spark is fast", "hello big data"])

words_map     = sentences.map(lambda s: s.split(" "))      # list of lists
words_flatmap = sentences.flatMap(lambda s: s.split(" "))  # flat list of words

#### `distinct()`, `union()`, `sample()`
- `distinct()` removes duplicates
- `union()` glues two RDDs together
- `sample()` takes a random fraction of the data

In [ ]:
dupes = sc.parallelize([1, 1, 2, 2, 3, 3, 3])

unique   = dupes.distinct()                       # remove duplicates
combined = numbers.union(dupes)                   # stack both RDDs
sampled  = numbers.sample(withReplacement=False, fraction=0.5, seed=42)

---
### 3.3 🎬 RDD Actions (they trigger the work!)

Only now does Spark actually execute the recipe. Each of these returns a normal Python value to your notebook (the *driver*).

| Action | What it returns |
|---|---|
| `collect()` | **All** elements as a Python list ⚠️ only for small data! |
| `take(n)` | The first `n` elements |
| `first()` | The first element |
| `count()` | Number of elements |
| `reduce(f)` | Combines all elements into one using `f` |
| `sum()`, `max()`, `min()`, `mean()` | Simple statistics |
| `countByValue()` | Dict of value → how many times it appears |
| `foreach(f)` | Runs `f` on each element (no return value) |

In [ ]:
print("collect :", squared.collect())      # ACTION: runs map, returns list
print("even    :", even.collect())         # ACTION: runs filter
print("take(3) :", numbers.take(3))        # first 3 items
print("first   :", numbers.first())        # just the first item
print("count   :", numbers.count())        # how many items

In [ ]:
print("sum     :", numbers.sum())
print("max     :", numbers.max())
print("min     :", numbers.min())
print("mean    :", numbers.mean())

# reduce: combine elements pairwise. Here we add them all up.
print("reduce  :", numbers.reduce(lambda a, b: a + b))

In [ ]:
print("map     :", words_map.collect())        # notice the nested lists
print("flatMap :", words_flatmap.collect())    # a flat list of words
print("distinct:", sorted(unique.collect()))
print("union   :", combined.collect())
print("sample  :", sampled.collect())
print("countByValue:", dict(dupes.countByValue()))

### 3.4 ⛓️ Chaining transformations

You normally chain many transformations together, then finish with **one action**. Spark runs the entire chain in one go.

**Goal:** take numbers 1 to 10, keep the even ones, square them, and add them up.

In [ ]:
result = (
    numbers
    .filter(lambda x: x % 2 == 0)   # transformation -> 2, 4, 6, 8, 10
    .map(lambda x: x * x)           # transformation -> 4, 16, 36, 64, 100
    .sum()                          # ACTION: everything actually runs here
)
print("Result:", result)

### 3.5 👀 Proof that Spark is lazy

Let's build a transformation that *prints* something each time it processes an element, and watch when the printing happens.

*(Tip: in local mode, `print` inside a function may show up in the terminal that launched Jupyter instead of the notebook. So we use a slow function and measure the time instead.)*

In [ ]:
import time

def slow_double(x):
    time.sleep(0.5)      # pretend this is heavy work
    return x * 2

small = sc.parallelize([1, 2, 3, 4], numSlices=4)

start = time.time()
doubled = small.map(slow_double)                       # transformation
print(f"After map()    : {time.time() - start:.2f} seconds  <- instant, nothing ran!")

start = time.time()
print(doubled.collect())                               # action
print(f"After collect(): {time.time() - start:.2f} seconds  <- the real work happens now")

✅ `map()` took ~0 seconds because it only **recorded** the step. `collect()` took time because that is when Spark really did the work.

### 3.6 🔑 Key-Value (Pair) RDDs

When each element is a `(key, value)` tuple, you unlock powerful transformations like `reduceByKey`, `groupByKey`, `sortByKey` and `join`.

These are **wide transformations**: Spark must move data between partitions so that all items with the same key end up together. This movement is called a **shuffle** (more in Part 5).

In [ ]:
# Classic word count, step by step
lines = sc.parallelize(["hello spark world", "spark is fast", "hello big data"])

pairs  = lines.flatMap(lambda l: l.split(" ")).map(lambda w: (w, 1))  # ("hello", 1), ("spark", 1)...
counts = pairs.reduceByKey(lambda a, b: a + b)                        # add the 1s per word

print("pairs :", pairs.collect())
print("counts:", counts.collect())

In [ ]:
# sortBy: order the results (here: most frequent word first)
top_words = counts.sortBy(lambda kv: kv[1], ascending=False)
print(top_words.take(3))

#### `reduceByKey` vs `groupByKey`

Both group by key, but:

- ✅ **`reduceByKey`** combines values *inside each partition first*, then shuffles the small results. Fast.
- ⚠️ **`groupByKey`** shuffles *every single value*, then groups. Slow and memory hungry.

**Rule of thumb: prefer `reduceByKey`.**

In [ ]:
scores = sc.parallelize([("alice", 10), ("bob", 5), ("alice", 20), ("bob", 15), ("carol", 7)])

print("reduceByKey:", scores.reduceByKey(lambda a, b: a + b).collect())
print("groupByKey :", scores.groupByKey().mapValues(list).collect())   # list of all values per key

#### `join`: combine two pair RDDs by key

In [ ]:
ages   = sc.parallelize([("alice", 30), ("bob", 25), ("carol", 41)])
cities = sc.parallelize([("alice", "Paris"), ("bob", "Rome"), ("dave", "Oslo")])

print("join      :", ages.join(cities).collect())            # only keys in BOTH
print("left join :", ages.leftOuterJoin(cities).collect())   # all keys from the left

## 4. 📊 Part B: DataFrames

A **DataFrame** is like a table (or a pandas DataFrame) with named columns, distributed over a cluster. It is the API you will use **most of the time** in real projects because:

- it is easier to read (column names, SQL-like operations)
- Spark's **Catalyst optimizer** can make it much faster than raw RDDs

The lazy rule is exactly the same: **transformations are lazy, actions run the job**.

### 4.1 Creating DataFrames

In [ ]:
employees = spark.createDataFrame(
    [
        (1, "Alice", "Engineering", 5000, 30),
        (2, "Bob",   "Engineering", 4000, 25),
        (3, "Carol", "Sales",       3500, 41),
        (4, "Dave",  "Sales",       3000, 35),
        (5, "Eve",   "HR",          3200, 29),
        (6, "Frank", "HR",          None, 50),   # a missing salary!
    ],
    schema=["id", "name", "dept", "salary", "age"],
)

departments = spark.createDataFrame(
    [("Engineering", "Paris"), ("Sales", "London"), ("Marketing", "Berlin")],
    schema=["dept", "city"],
)

### 4.2 🎬 First actions: look at your data

- `show(n)` prints a nice table (default 20 rows)
- `printSchema()` shows column names and types *(this one is not really an action: it only reads metadata)*
- `count()` returns the number of rows

In [ ]:
employees.show()             # ACTION: prints the table
employees.printSchema()      # column names and types
print("Rows:", employees.count())   # ACTION
print("Columns:", employees.columns)

---
### 4.3 🔧 DataFrame Transformations (lazy)

Import the helper functions module as `F`. It contains `col`, `lit`, `when`, `avg`, `sum`, and more.

In [ ]:
from pyspark.sql import functions as F

#### `select`: pick columns

In [ ]:
employees.select("name", "salary").show()     # transformation + show() action

#### `filter` / `where`: pick rows
Both names do exactly the same thing.

In [ ]:
employees.filter(F.col("salary") > 3500).show()

# Combine conditions with & (and), | (or), ~ (not). Wrap each condition in parentheses!
employees.where((F.col("dept") == "Engineering") & (F.col("age") < 28)).show()

#### `withColumn`: add or replace a column

In [ ]:
with_bonus = employees.withColumn("bonus", F.col("salary") * 0.10)
with_bonus.select("name", "salary", "bonus").show()

#### `when / otherwise`: if-else logic for columns

In [ ]:
labelled = employees.withColumn(
    "level",
    F.when(F.col("salary") >= 4000, "high")
     .when(F.col("salary") >= 3200, "medium")
     .otherwise("low")           # also catches the missing salary (None)
)
labelled.select("name", "salary", "level").show()

#### `withColumnRenamed` and `drop`

In [ ]:
employees.withColumnRenamed("dept", "department").drop("age").show()

#### `orderBy` / `sort`: sort rows

In [ ]:
employees.orderBy(F.col("salary").desc()).show()   # highest salary first (nulls go last)

#### Handling missing values: `dropna`, `fillna`

In [ ]:
employees.dropna().show()                 # remove rows that contain any null
employees.fillna({"salary": 0}).show()    # replace nulls with a default value

#### `distinct` and `dropDuplicates`

In [ ]:
employees.select("dept").distinct().show()

#### `groupBy` + `agg`: summarise per group
This is a **wide transformation** (it shuffles data so each group ends up together).

In [ ]:
employees.groupBy("dept").agg(
    F.count("*").alias("people"),
    F.avg("salary").alias("avg_salary"),
    F.max("age").alias("oldest"),
).show()

#### `join`: combine two DataFrames

| Join type | Meaning |
|---|---|
| `inner` (default) | Only departments present in **both** tables |
| `left` | Everything from the left table, matches from the right |
| `right` | Everything from the right table |
| `outer` | Everything from both |

In [ ]:
print("Inner join:")
employees.join(departments, on="dept", how="inner").select("name", "dept", "city").show()

print("Left join (HR has no city, so it is null):")
employees.join(departments, on="dept", how="left").select("name", "dept", "city").show()

#### `union`: stack two DataFrames with the same columns

In [ ]:
new_hire = spark.createDataFrame([(7, "Grace", "Sales", 3300, 27)], schema=employees.columns)
employees.union(new_hire).show()

#### Using SQL instead

If you prefer SQL, register the DataFrame as a *temporary view*. The result of `spark.sql(...)` is a DataFrame, so it is a lazy transformation too.

In [ ]:
employees.createOrReplaceTempView("employees")

spark.sql('''
    SELECT dept, ROUND(AVG(salary), 1) AS avg_salary
    FROM employees
    GROUP BY dept
    ORDER BY avg_salary DESC
''').show()

---
### 4.4 🎬 DataFrame Actions

| Action | What it does |
|---|---|
| `show(n)` | Prints the first `n` rows as a table |
| `count()` | Number of rows |
| `collect()` | All rows as a list of `Row` objects ⚠️ small data only |
| `take(n)` / `head(n)` | First `n` rows as a list |
| `first()` | The first row |
| `toPandas()` | Converts to a pandas DataFrame ⚠️ small data only |
| `write...` | Saves data to disk (CSV, Parquet, JSON...) |

In [ ]:
rows = employees.collect()          # ACTION: list of Row objects
print(rows[0])
print(rows[0]["name"], rows[0].salary)   # access by name or attribute

print(employees.first())            # ACTION: just the first row
print(employees.take(2))            # ACTION: first 2 rows

In [ ]:
pdf = employees.toPandas()          # ACTION: bring the data into pandas
pdf.head()

#### Writing data to disk

Writing is an action too. Spark writes a **folder** of part-files (one per partition), not a single file.

In [ ]:
(
    employees.write
    .mode("overwrite")               # replace the folder if it already exists
    .parquet("output/employees_parquet")
)
print("Saved! Now reading it back:")
spark.read.parquet("output/employees_parquet").show(3)

### 4.5 🔍 See the plan Spark builds: `explain()`

`explain()` shows the plan **without running it**. This is the "waiter's notepad" from our restaurant analogy.

Read it **from the bottom up**: bottom = reading the data, top = the final result.

In [ ]:
plan = (
    employees
    .filter(F.col("salary") > 3000)
    .groupBy("dept")
    .agg(F.avg("salary").alias("avg_salary"))
)

plan.explain()      # shows the plan only. No job was run yet!

Look for the word **`Exchange`** in the plan: that is a **shuffle**. It appears because of `groupBy`. Let's talk about that next.

## 5. 🔀 Narrow vs Wide Transformations

Transformations come in two flavours. This matters a lot for performance.

| | **Narrow** | **Wide** |
|---|---|---|
| Data movement | Each output partition depends on **one** input partition | Output partitions depend on **many** input partitions |
| Network shuffle? | ❌ No, fast | ✅ Yes, expensive |
| Examples | `map`, `filter`, `flatMap`, `select`, `withColumn`, `union` | `groupBy`, `reduceByKey`, `join`, `distinct`, `orderBy`, `repartition` |

```
NARROW  (filter):                 WIDE  (groupBy):
 [P1] ──► [P1']                    [P1] ─┬─► [P1']
 [P2] ──► [P2']                    [P2] ─┼─► [P2']
 [P3] ──► [P3']                    [P3] ─┴─► [P3']     (every partition talks to every other)
```

**Why care?** Spark splits a job into **stages** at each shuffle. Fewer shuffles = faster jobs. A good habit is to **filter early** (narrow) before doing wide operations.

In [ ]:
# Good habit: filter and select BEFORE the expensive groupBy/join,
# so less data is shuffled.
efficient = (
    employees
    .filter(F.col("dept") != "HR")      # narrow: throw rows away early
    .select("dept", "salary")           # narrow: keep only needed columns
    .groupBy("dept")                    # wide: shuffle happens on less data
    .agg(F.sum("salary").alias("total_salary"))
)
efficient.show()

## 6. 💾 Caching: don't repeat the work

Because of lazy evaluation, **every action recomputes the whole chain from scratch**.

```python
df = expensive_transformations(...)
df.count()   # computes everything
df.show()    # computes everything AGAIN
```

If you reuse a dataset several times, tell Spark to keep it in memory with `cache()` (or `persist()` for more control).

In [ ]:
import time

def slow_plus_one(x):
    time.sleep(0.2)
    return x + 1

data = sc.parallelize(range(10), numSlices=2).map(slow_plus_one)

# WITHOUT cache: each action redoes the slow work
t = time.time(); data.count(); print(f"1st action (no cache): {time.time()-t:.2f}s")
t = time.time(); data.count(); print(f"2nd action (no cache): {time.time()-t:.2f}s")

In [ ]:
data_cached = data.cache()      # mark it to be kept in memory (still lazy!)

t = time.time(); data_cached.count(); print(f"1st action (cache filling): {time.time()-t:.2f}s")
t = time.time(); data_cached.count(); print(f"2nd action (from cache)   : {time.time()-t:.2f}s  <- much faster!")

data_cached.unpersist()         # free the memory when you are done

> 💡 `cache()` itself is lazy. The data is only stored **the first time an action runs**. Use it when the same dataset feeds several actions, and `unpersist()` when finished.

## 7. 🏁 Mini Project: Word Count, End to End

The "Hello World" of big data. Let's combine everything we learned. We'll count the most common words in some text using both RDDs and DataFrames.

In [ ]:
text = [
    "Spark is fast and Spark is simple",
    "Simple code is good code",
    "Fast code is even better",
]

**Version 1: RDD**

In [ ]:
rdd_result = (
    sc.parallelize(text)
    .flatMap(lambda line: line.lower().split(" "))   # T: split lines into words
    .map(lambda word: (word, 1))                     # T: pair each word with 1
    .reduceByKey(lambda a, b: a + b)                 # T: add up the 1s per word (shuffle)
    .sortBy(lambda kv: kv[1], ascending=False)       # T: most frequent first
    .take(5)                                         # A: the ONLY action, triggers everything
)
print(rdd_result)

**Version 2: DataFrame** (shorter, and optimised by Spark)

In [ ]:
lines_df = spark.createDataFrame([(t,) for t in text], schema=["line"])

(
    lines_df
    .select(F.explode(F.split(F.lower("line"), " ")).alias("word"))  # T: one row per word
    .groupBy("word")                                                 # T: group same words
    .count()                                                         # T: count per group
    .orderBy(F.col("count").desc())                                  # T: sort
    .show(5)                                                         # A: show the top 5
)

## 8. 📝 Cheat Sheet

### 🔧 Transformations (lazy, return a new dataset)

| | RDD | DataFrame |
|---|---|---|
| **Narrow** | `map`, `filter`, `flatMap`, `union`, `sample` | `select`, `filter`/`where`, `withColumn`, `drop`, `withColumnRenamed`, `union` |
| **Wide** (shuffle) | `reduceByKey`, `groupByKey`, `sortBy`, `distinct`, `join` | `groupBy().agg()`, `join`, `orderBy`, `distinct`, `repartition` |

### 🎬 Actions (trigger the computation)

| | RDD | DataFrame |
|---|---|---|
| **Get data back** | `collect`, `take`, `first` | `collect`, `take`/`head`, `first`, `toPandas` |
| **Compute a value** | `count`, `sum`, `max`, `min`, `mean`, `reduce`, `countByValue` | `count` |
| **Display** | - | `show` |
| **Save** | `saveAsTextFile` | `write.parquet/csv/json` |
| **Side effects** | `foreach` | `foreach` |

### 🧭 Golden Rules

1. **Nothing happens until an action is called.**
2. **Datasets are immutable:** transformations return *new* datasets.
3. **Filter early, shuffle late:** reduce data before wide transformations.
4. **Avoid `collect()` on big data:** it pulls everything into the driver's memory. Prefer `take()`, `show()` or `write`.
5. **Use `cache()`** when you reuse a dataset across several actions.
6. **Prefer DataFrames** over RDDs for most work.
7. **Prefer `reduceByKey`** over `groupByKey`.

## 9. 🏋️ Practice Exercises

Try these yourself. Solutions are in the next cell (no peeking! 😄).

**Exercise 1 (RDD):** From the numbers 1 to 20, keep multiples of 3, multiply each by 10, and return the list.

**Exercise 2 (RDD):** Given `["a b a", "b c", "a"]`, find how many times each letter appears.

**Exercise 3 (DataFrame):** Using `employees`, find the average salary of people **older than 28**, per department, sorted from highest to lowest.

**Exercise 4 (DataFrame):** Add a column `is_senior` that is `True` when `age >= 35`, then count how many seniors there are.

In [ ]:
# ---------------- SOLUTIONS ----------------

# Exercise 1
print(sc.parallelize(range(1, 21)).filter(lambda x: x % 3 == 0).map(lambda x: x * 10).collect())

# Exercise 2
print(
    sc.parallelize(["a b a", "b c", "a"])
    .flatMap(lambda s: s.split(" "))
    .map(lambda l: (l, 1))
    .reduceByKey(lambda a, b: a + b)
    .collect()
)

# Exercise 3
(
    employees
    .filter(F.col("age") > 28)
    .groupBy("dept")
    .agg(F.avg("salary").alias("avg_salary"))
    .orderBy(F.col("avg_salary").desc())
    .show()
)

# Exercise 4
seniors = employees.withColumn("is_senior", F.col("age") >= 35)
print("Seniors:", seniors.filter("is_senior").count())

## 10. 🧹 Clean Up

Always stop your Spark session when you are finished, to free up resources.

In [ ]:
spark.stop()

---
🎉 **Congratulations!** You now understand the heart of Spark: *transformations build the plan, actions run it.*

**Where to go next:** Spark SQL functions, window functions, reading CSV/JSON files, partitioning strategies, and Structured Streaming. Official docs: <https://spark.apache.org/docs/latest/>